In [ ]:
!pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data=pd.read_csv('samsum-train.csv')
val_data=pd.read_csv('samsum-validation.csv')

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


There are lot of unnecessary tag and word are present we need to remove that


In [ ]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

Now for Simple computation we took 4K data from train and 500 sample from val

In [ ]:
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)


In [ ]:
print("Train size=",train_data.shape)
print("Val_size=",val_data.shape)

Train size= (4000, 3)
Val_size= (500, 3)


**Text Preprocessing**

In [ ]:
import re
def clean_txt(text):
  text=re.sub(r"\r\n"," ") #removing space
  text=re.sub(r"\s+"," ") #Removing the space
  text=re.sub(r"<.*?>"," ") #removing the tag
  text=text.strip().lower() #remove the outer space and Make it into lowercase
  return text


In [ ]:
import re

def clean_txt(text):
  text=re.sub(r"\r\n"," ", text) #removing space
  text=re.sub(r"\s+"," ", text) #Removing the space
  text=re.sub(r"<.*?>"," ", text) #removing the tag
  text=text.strip().lower() #remove the outer space and Make it into lowercase
  return text

train_data['dialogue']=train_data['dialogue'].apply(clean_txt)
train_data['summary']=train_data['summary'].apply(clean_txt)

val_data['dialogue']=val_data['dialogue'].apply(clean_txt)
val_data['summary']=val_data['summary'].apply(clean_txt)


In [ ]:
train_data['dialogue'][2]

"ian: god damn! i'm not gonna make it on time! phil: what happened? ian: long story. i'll be there in about an hour. phil: so, we'll begin the meeting without you? ian: certainly, you do that. phil: should i go into details? ian: no, wait for me. just give them the overall picture. phil: no problem. drive safely!"

Tokenizer

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
#Raw-data=>fine tune input Token

def tokenize(data):
  inputs=tokenizer(data['dialogue'],padding="max_length",max_length=512,truncation=True)
  target=tokenizer(data['summary'],padding="max_length",max_length=150,truncation=True)

  inputs['labels']=target['inputs_id']
  return inputs

In [ ]:
# Redefine tokenize function with the fix for 'inputs_id'
def tokenize(data):
  inputs=tokenizer(data['dialogue'],padding="max_length",max_length=512,truncation=True)
  target=tokenizer(data['summary'],padding="max_length",max_length=150,truncation=True)

  inputs['labels']=target['input_ids'] # Corrected from 'inputs_id' to 'input_ids'
  return inputs

train_set=train_data.apply(tokenize,axis=1).tolist()
val_set=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_set

[{'input_ids': [3, 29, 9, 6736, 10, 3, 23, 31, 51, 773, 9, 19682, 3, 29, 9, 6736, 10, 103, 25, 43, 136, 126, 21705, 24, 25, 54, 1568, 58, 2662, 1050, 10, 6865, 3, 10, 61, 2662, 1050, 10, 131, 5607, 140, 125, 773, 13, 21705, 33, 25, 1638, 16, 58, 3, 29, 9, 6736, 10, 424, 659, 11, 6613, 3, 29, 9, 6736, 10, 1066, 4092, 42, 11043, 3803, 3, 29, 9, 6736, 10, 59, 3, 9, 600, 1819, 13, 17201, 18, 89, 23, 2662, 1050, 10, 410, 25, 1605, 959, 45, 48, 1590, 87, 210, 3870, 5818, 58, 3, 29, 9, 6736, 10, 59, 780, 2662, 1050, 10, 207, 6, 24, 3231, 178, 28, 128, 1245, 931, 2662, 1050, 10, 166, 13, 66, 1077, 82, 1305, 126, 1764, 96, 6279, 97, 3, 23, 530, 3, 60, 18860, 920, 38, 3, 9, 12593, 15, 121, 2662, 1050, 10, 8, 2233, 845, 66, 81, 34, 3, 29, 9, 6736, 10, 24, 31, 7, 3, 9, 17056, 564, 2662, 1050, 10, 168, 17945, 68, 8, 21705, 19, 248, 2662, 1050, 10, 34, 65, 128, 19752, 8073, 68, 167, 13, 8, 97, 34, 11331, 7, 66, 13, 39, 5598, 2662, 1050, 10, 659, 6, 6613, 11, 11043, 1898, 3, 29, 9, 6736, 10, 2993, 14

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Load Model

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")
model.to(device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
train_args=TrainingArguments(
    output_dir="./result",
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [ ]:
trainer=Trainer(
    model=model,
    args=train_args,
    train_dataset=train_set,
    eval_dataset=val_set
)

Train The Model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("./save_model")
tokenizer.save_pretrained('.save_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('.save_model/tokenizer_config.json', '.save_model/tokenizer.json')

In [ ]:
def summarize_text(dialogue):
  dialogue = clean_txt(dialogue) #text preprocess
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  )
  model.to(device)
  #generate the op=>
  target = model.generate(
      input_ids=inputs['input_ids'].to(device),
      attention_mask=inputs['attention_mask'].to(device),
      max_length=150,
      num_beams=4,
      early_stopping=True
  )
  #decode the generate value
  summary=tokenizer.decode(target[0], skip_special_tokens=True)
  return summary

In [ ]:
str=input()
res=summarize_text(str)
print(res)

I am a B.Tech student with a strong interest in technology and problem-solving. Currently, I am learning subjects like Data Structures and Artificial Intelligence to improve my skills and knowledge. I enjoy coding and building projects, and I always try to learn new things that can help me grow in the tech field. I am also interested in creating useful platforms, such as a library website for students to access notes and previous year questions easily. My goal is to become a skilled software developer and contribute to innovative and impactful projects in the future.
i am a b.tech student with a strong interest in technology and problem-solving. currently, i am learning subjects such as data structures and artificial intelligence to improve my skills and knowledge. i am also interested in creating useful platforms for students to access notes and previous year questions easily.


In [ ]:
from google.colab import files
import shutil

shutil.make_archive('save_model', 'zip', '/content/save_model')

files.download('save_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>